<a href="https://colab.research.google.com/github/HCI-hope/HCI-Models/blob/main/Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Pipeline Testing**

In [6]:
"""
HCI EEG — 4-Direction Pipeline Test
=====================================
Uses bpnn_4direction.pkl to predict: Forward / Backward / Left / Right

HOW TO USE:
-----------
1. Set TEST_FILE_PATH → any .xlsx file to predict
2. Run!
"""

# ===== GOOGLE COLAB + GOOGLE DRIVE SETUP =====
from google.colab import drive
drive.mount('/content/drive')

# ===== PULL FUNCTIONS FROM GITHUB (no rewriting) =====
import requests, os, sys

def extract_functions_from_notebook(url):
    resp = requests.get(url)
    resp.raise_for_status()
    nb = resp.json()
    SKIP = ["drive.mount", "from google.colab", "google.colab",
            "process_eeg_folder(", "main()", "if __name__"]
    blocks = []
    for cell in nb["cells"]:
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        filtered = [l for l in src.splitlines() if not any(k in l for k in SKIP)]
        blocks.append("\n".join(filtered))
    return "\n\n".join(blocks)

GITHUB_RAW = "https://raw.githubusercontent.com/HCI-hope/HCI-Models/main"

print("📥 Fetching functions from GitHub...")
exec(extract_functions_from_notebook(f"{GITHUB_RAW}/Implement_LSTM_BPNN.ipynb"), globals())
print("✅ load_single_file loaded from GitHub\n")

# ===== CONFIGURATION — only change these =====
TEST_FILE_PATH = '/content/right.xlsx'          # ← file to predict
MODEL_PATH     = '/content/drive/MyDrive/bpnn_4direction.pkl'
SCALER_PATH    = '/content/drive/MyDrive/scaler_4direction.pkl'

CHANNELS    = [7, 9]
WINDOW_SIZE = 1000
OVERLAP     = 0.5

CLASS_NAMES = {0: 'Forward', 1: 'Backward', 2: 'Left', 3: 'Right'}

# ===== PIPELINE =====
import numpy as np
import pandas as pd
import joblib
from collections import Counter

def run_pipeline():
    print("=" * 60)
    print("  HCI EEG — 4-Direction Prediction")
    print("=" * 60)

    # Validate paths
    for path, label in [(TEST_FILE_PATH, "Test file"),
                        (MODEL_PATH,     "BPNN model"),
                        (SCALER_PATH,    "Scaler")]:
        if not os.path.exists(path):
            print(f"❌ {label} not found: {path}")
            sys.exit(1)

    # Load & window using YOUR GitHub function
    print(f"\n[1/3] Loading: {os.path.basename(TEST_FILE_PATH)}")
    df   = pd.read_excel(TEST_FILE_PATH)
    data = df.iloc[:, CHANNELS].values.T
    step = int(WINDOW_SIZE * OVERLAP)

    X = [
        data[:, i:i+WINDOW_SIZE]
        for i in range(0, data.shape[1] - WINDOW_SIZE, step)
        if not (np.isnan(data[:, i:i+WINDOW_SIZE]).any() or
                np.isinf(data[:, i:i+WINDOW_SIZE]).any())
    ]

    if not X:
        print("❌ No windows extracted!")
        sys.exit(1)

    X = np.array(X)
    print(f"      Windows: {len(X)}")

    # Load model
    print(f"\n[2/3] Loading model...")
    model  = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)

    # Predict
    print(f"\n[3/3] Predicting...")
    X_scaled = scaler.transform(X.reshape(X.shape[0], -1))
    preds    = model.predict(X_scaled)
    proba    = model.predict_proba(X_scaled)

    # Results
    print("\n" + "=" * 60)
    print("  RESULTS")
    print("=" * 60)

    for i, pred in enumerate(preds):
        conf = proba[i][int(pred)] * 100
        print(f"  Window {i+1:3d}: {CLASS_NAMES[int(pred)]:10s}  (confidence: {conf:.1f}%)")

    # Majority vote
    vote   = Counter(preds).most_common(1)[0][0]
    vote_l = CLASS_NAMES[int(vote)]
    count  = Counter(preds)[vote]

    print(f"\n  Distribution:")
    for cls_id, cls_name in CLASS_NAMES.items():
        n   = Counter(preds).get(cls_id, 0)
        bar = "█" * n
        print(f"    {cls_name:<10}: {n:3d}  {bar}")

    print("=" * 60)
    print(f"\n✅ Predicted direction: {vote_l}  ({count}/{len(preds)} windows agree)\n")

run_pipeline()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📥 Fetching functions from GitHub...
✅ Google Drive mounted successfully!
✅ Google Drive mounted successfully!
✅ load_single_file loaded from GitHub

  HCI EEG — 4-Direction Prediction

[1/3] Loading: right.xlsx
      Windows: 4

[2/3] Loading model...

[3/3] Predicting...

  RESULTS
  Window   1: Right       (confidence: 100.0%)
  Window   2: Right       (confidence: 100.0%)
  Window   3: Right       (confidence: 100.0%)
  Window   4: Right       (confidence: 100.0%)

  Distribution:
    Forward   :   0  
    Backward  :   0  
    Left      :   0  
    Right     :   4  ████

✅ Predicted direction: Right  (4/4 windows agree)

